In [ ]:
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from lxml import etree
import time
from datetime import date
import ast

user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:148.0) Gecko/20100101 Firefox/148.0"
profile_path = r"C:\\Users\\aranw\AppData\\Roaming\\Mozilla\\Firefox\\Profiles\\a5jhw6zg.default"
options = Options()
options.add_argument("-profile")
options.add_argument(profile_path)
options.add_argument("--headless")


score_site = "https://scores.vicriichi.org/events/boxhill"
starting_pts = 30000
uma = {"first": 30,
       "second": 10,
       "third": -10,
       "fourth": -30}
chombo = -20

In [2]:
def calc_score(score):
    return (score.get("points") - starting_pts)/1000 + uma.get(score.get("placement"))

def calc_points_needed(final_pts, placement):
    return (final_pts - uma.get(placement))*1000 + starting_pts

In [ ]:
game_list = []
browser = webdriver.Firefox(options=options)
browser.get(score_site)
time.sleep(5)
html = browser.page_source
soup = BeautifulSoup(html, 'lxml')
for div in soup.find_all("div", class_="game-display"):
    game_list.append(div)

game_list = game_list[::-1]

In [6]:
game_list

[<div class="game-display"><div><div><form><div>Game #1</div></form></div><div>11/1/2025</div></div><div class="game-player-display"><div>Max Peng</div><div>42300</div><div style="font-weight:bold;color:#55AB00FF">42.3</div></div><div class="game-player-display"><div>Sokhun Lim</div><div>38900</div><div style="font-weight:bold;color:#55AB00FF">18.9</div></div><div class="game-player-display"><div>Josh Tey</div><div>23700</div><div style="font-weight:bold;color:red">-16.3</div></div><div class="game-player-display"><div>Aaron Chee</div><div>15100</div><div style="font-weight:bold;color:red">-44.9</div></div></div>,
 <div class="game-display"><div><div><form><div>Game #2</div></form></div><div>11/1/2025</div></div><div class="game-player-display"><div>Ivan Palko</div><div>46600</div><div style="font-weight:bold;color:#55AB00FF">46.6</div></div><div class="game-player-display"><div>Jake Beltrami</div><div>32900</div><div style="font-weight:bold;color:#55AB00FF">12.9</div></div><div class=

In [ ]:
def compile_results(game_list):
    results_df = pd.DataFrame(columns=['Game #',
                      'Date',
                      '1',
                      '2',
                      '3',
                      '4'])
    
    for game in game_list:
        game_data = {}
        header_spans = game.find("div", class_="game-display-header").find_all("span")
        game_title = header_spans[0].text
        game_date = header_spans[1].text

        player_divs = game.find_all("div", class_="game-player-display")
        for i, player in enumerate(player_divs):
            cells = player.find_all("div")
            results = {"Name": cells[0].text,
                    "Points": int(cells[1].text),
                    "Score": float(cells[2].text)}
            game_data[f"{i+1}"] = [results]
        
        game_data["Game #"] = game_title
        game_data["Date"] = game_date
        results_df = pd.concat([results_df, pd.DataFrame(game_data)], ignore_index=True)
    
    return results_df

In [8]:
results_df = compile_results(game_list)

In [9]:
results_df.to_csv(f'results-{date.today()}.csv', index=False)

In [10]:
def import_results(file, cols):
    df = pd.read_csv(file)

    for col in cols:
        df[col] = df[col].apply(ast.literal_eval)
        
    return df

In [11]:
results_df = import_results("./results-2026-09-05.csv", ['1','2','3','4'])

# Stats Calculations

In [12]:
def calc_player_trends(df):
    player_score_trends = {}
    player_stats = {}
    for i in range(len(df)):
        for j in range(2,6):
            result = df.iloc[i,j]
            player_name = result.get('Name')
            player_score = result.get('Score')
            player_ranking = f'{j-1}'
            
            player_score_trends.setdefault(player_name, []).append(player_score)
            player_stats.setdefault(player_name, {'1':0, '2':0, '3':0, '4':0})[player_ranking] += 1
    return player_score_trends, player_stats

In [13]:
player_score_trends, player_stats = calc_player_trends(results_df)

In [14]:
players = list(player_score_trends.keys())

In [23]:
def calc_best_ten(results, window_size=10, limit=0):
    max = -999
    max_frame = []
    if limit > 0:
        results = results[-limit:]
    
    for i in range(len(results)-window_size+1):
        frame = results[i:i+window_size]
        sum_frame = np.sum(frame)
        if sum_frame > max:
            max = sum_frame
            max_frame = frame
            
        if i == len(results) - window_size:
            print(f"Max Frame: {max_frame}")
            diff = max - sum_frame
            if diff == 0:
                print("Congrats, you're on your high run!")
            else:
                print(f"You need to score higher than {diff:.1f} on your next game to beat your PB")


In [24]:
def translate_stats(stats):
    num_games = sum(list(stats.values()))
    first = stats['1']
    second = stats['2']
    third = stats['3']
    fourth = stats['4']
    print(f'Total Games: {num_games}')
    print(f'First: {first/num_games*100:.2f}% ({first})')
    print(f'Second: {second/num_games*100:.2f}% ({second})')
    print(f'Third: {third/num_games*100:.2f}% ({third})')
    print(f'Fourth: {fourth/num_games*100:.2f}% ({fourth})')

In [28]:
def get_player_summary(player_score_trends, player_stats, player, limit=0):
    print(f'Summary for {player}:')
    calc_best_ten(player_score_trends.get(player), limit=limit)
    print()
    print('Ranking Distribution:')
    translate_stats(player_stats.get(player))
    

In [29]:
get_player_summary(player_score_trends, player_stats, 'Xiran', limit=18)

Summary for Xiran:
Max Frame: [10.5, 39.1, 48.4, 47.6, -28.4, -57.2, 11.9, 70.6, -19.0, 14.4]
You need to score higher than 2.6 on your next game to beat your PB

Ranking Distribution:
Total Games: 88
First: 20.45% (18)
Second: 22.73% (20)
Third: 32.95% (29)
Fourth: 23.86% (21)
